# 11 — Tiga backbone di ImageNet val (GPU)

**Kenapa notebook ini ada.** `topvenue_gap_audit.md` A2: klaim PCC berdiri di dua
backbone, sedangkan paper lama punya empat. Aturan keras `AGENTS.md` melarang
membandingkan lintas backbone, jadi menambah backbone berarti **menghitung ulang**
semuanya di atasnya — dan itu murah kalau yang disimpan adalah logitnya.

Yang dihasilkan per backbone, dan tidak lebih dari itu:

| Berkas | Untuk apa |
|---|---|
| `logits.npy` | agar suhu/kalibrasi bisa diubah nanti **tanpa GPU** |
| `scores.npy` | softmax — format dump yang sudah dipakai notebook 06/09 |
| `labels.npy` | label ImageFolder, urutan WNID |
| `fc_weight.npy`, `fc_bias.npy` | **φ kepala** — satu-satunya keluarga φ yang pernah lolos |
| `meta.json` | akurasi, transform, nama lapisan kepala |

**ResNet-50 ikut diekstrak ulang di sini**, walaupun notebook 07 sudah punya logitnya.
Notebook 07 memakai protokol 25.000 cal / 25.000 test milik paper UM-TTA; notebook ini
memakai 50.000 penuh seperti dump CCC. Mencampur keduanya di satu tabel backbone akan
membandingkan protokol, bukan backbone.

**~10 menit GPU.** Yang lama justru unduhan val 6,7 GB.

## 1. Config

In [ ]:
# === EDIT ME ===========================================================
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
UMTTA_URL  = 'https://github.com/octadion/umtta_conformal.git'
UMTTA_DIR  = '/content/umtta'

VAL_TAR = 'https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar'
LBL_URL = ('https://github.com/tensorflow/models/raw/master/research/slim/'
           'datasets/imagenet_2012_validation_synset_labels.txt')
VAL_DIR = '/content/imagenet_val'

# akurasi top-1 terbit torchvision (bobot IMAGENET1K_V1). Dipakai sebagai
# pemeriksaan, bukan hiasan: transform yang salah menurunkannya beberapa persen
# dan tidak ada gejala lain yang muncul.
BACKBONES = [('resnet50',      0.76130),
             ('vit_b_16',      0.81072),
             ('convnext_tiny', 0.82520)]
ACC_TOL = 0.02
BATCH_SIZE = 128

SAVE_ROOT = '/content/results/backbones'
DEST_ROOT = DRIVE_ROOT + '/backbones'
# =======================================================================
print('backbone:', [b for b, _ in BACKBONES])

## 2. Mount, repo, GPU

In [ ]:
import os, subprocess, sys, glob, time, json, shutil
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DEST_ROOT, exist_ok=True)

if not os.path.isdir(UMTTA_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', UMTTA_URL, UMTTA_DIR], check=True)
subprocess.run(['pip', 'install', '-q', '-r',
                os.path.join(UMTTA_DIR, 'requirements.txt')], check=False)
if UMTTA_DIR not in sys.path:
    sys.path.insert(0, UMTTA_DIR)

import torch, numpy as np
import torch.nn as nn
import torch.utils.data as tdata
import torchvision
import torchvision.transforms as transforms
assert torch.cuda.is_available(), 'notebook ini butuh GPU'
print('torch', torch.__version__, '| torchvision', torchvision.__version__,
      '|', torch.cuda.get_device_name(0))

# konstruksi model diambil dari repo supaya identik dengan paper lama
from multimodel_eval import load_model
print('load_model terimpor dari multimodel_eval')

## 3. Val bersih — unduh dan tata ke folder WNID

Identik dengan notebook 07 dan 10. Hasilnya di-assert.

In [ ]:
os.makedirs('/content/dl', exist_ok=True)
TAR = '/content/dl/ILSVRC2012_img_val.tar'
LBL = '/content/dl/val_synset_labels.txt'

if not os.path.exists(LBL) or os.path.getsize(LBL) < 100000:
    subprocess.run(['wget', '-q', LBL_URL, '-O', LBL], check=True)
lbls = [l.strip() for l in open(LBL) if l.strip()]
assert len(lbls) == 50000, 'label harus 50.000 baris, dapat ' + str(len(lbls))

if len(glob.glob(VAL_DIR + '/*/*.JPEG')) < 50000:
    if not os.path.exists(TAR) or os.path.getsize(TAR) < 6_000_000_000:
        subprocess.run(['apt-get', 'install', '-qq', 'aria2'], check=False)
        t0 = time.time()
        subprocess.run(['aria2c', '-x', '16', '-s', '16', VAL_TAR,
                        '-d', '/content/dl', '-o', 'ILSVRC2012_img_val.tar'],
                       check=True)
        print('unduh val {:.0f}s | {:.2f} GB'.format(
            time.time() - t0, os.path.getsize(TAR) / 1e9))
    RAW = '/content/dl/raw_val'
    os.makedirs(RAW, exist_ok=True)
    if len(glob.glob(RAW + '/*.JPEG')) < 50000:
        subprocess.run(['tar', '-xf', TAR, '-C', RAW], check=True)
    for w in set(lbls):
        os.makedirs(os.path.join(VAL_DIR, w), exist_ok=True)
    for i, w in enumerate(lbls, start=1):
        src = os.path.join(RAW, 'ILSVRC2012_val_%08d.JPEG' % i)
        if os.path.exists(src):
            os.rename(src, os.path.join(VAL_DIR, w, os.path.basename(src)))
    for p in (TAR, RAW):
        subprocess.run(['rm', '-rf', p], check=False)

dirs = sorted(d for d in os.listdir(VAL_DIR)
              if os.path.isdir(os.path.join(VAL_DIR, d)))
counts = [len(os.listdir(os.path.join(VAL_DIR, d))) for d in dirs]
assert len(dirs) == 1000 and sum(counts) == 50000
assert min(counts) == max(counts) == 50
print('val bersih OK: 1000 kelas x 50 = 50.000')

## 4. Transform per backbone — DIAMBIL dari bobotnya, bukan disamakan

Ini satu-satunya tempat notebook ini bisa salah tanpa gejala. `resnet50` dan
`vit_b_16` memakai resize 256 → crop 224, tetapi bobot `convnext_tiny` torchvision
memakai **resize 236** → crop 224. Memakai satu transform untuk ketiganya akan
menurunkan akurasi ConvNeXt sekitar satu persen — cukup untuk merusak angka, terlalu
kecil untuk terlihat mencurigakan.

Jadi transform-nya **dibaca dari metadata bobotnya sendiri** bila torchvision cukup
baru, dan yang benar-benar dipakai dicatat di `meta.json`.

In [ ]:
FALLBACK = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

def transform_for(name):
    """(transform, deskripsi) untuk satu backbone."""
    try:
        w = torchvision.models.get_model_weights(name)['IMAGENET1K_V1']
        return w.transforms(), 'weights.transforms() ' + str(w)
    except Exception as e:
        print('  transform bawaan tak tersedia ({}), pakai 256/224'.format(
            type(e).__name__))
        return FALLBACK, 'fallback resize256/crop224'

for nm, _ in BACKBONES:
    t, d = transform_for(nm)
    print('  {:14s} {}'.format(nm, d.replace(chr(10), ' ')[:90]))

## 5. Kepala klasifikasi — DITEMUKAN, bukan ditebak

`fc` untuk ResNet, `heads.head` untuk ViT, `classifier[2]` untuk ConvNeXt. Menghafal
tiga jalur atribut itu cara yang rapuh untuk gagal diam-diam kalau torchvision
berubah, jadi yang dicari adalah **`nn.Linear` terakhir dengan 1000 keluaran** dan
namanya dicetak untuk diperiksa mata.

In [ ]:
def find_head(model):
    """(nama, modul) lapisan Linear terakhir dengan out_features == 1000."""
    hits = [(n, m) for n, m in model.named_modules()
            if isinstance(m, nn.Linear) and m.out_features == 1000]
    assert hits, 'tidak ada Linear dengan 1000 keluaran -- kepala tidak dikenali'
    return hits[-1]

print('pencari kepala siap')

## 6. Ekstraksi

Satu lintasan maju per backbone atas 50.000 citra. Tidak ada TTA, tidak ada
augmentasi — dump ini yang menjadi dasar semua eksperimen konformal berikutnya, dan
semuanya jalan tanpa GPU setelah ini.

Backbone yang dumpnya sudah utuh di Drive dilewati, jadi notebook ini bisa dijalankan
ulang setelah runtime mati tanpa mengulang apa pun.

In [ ]:
import gc
META = {}

def files_of(nm):
    return ['logits.npy', 'scores.npy', 'labels.npy', 'fc_weight.npy',
            'fc_bias.npy', 'meta.json']

def in_drive(nm):
    d = os.path.join(DEST_ROOT, nm)
    return all(os.path.exists(os.path.join(d, f)) and
               os.path.getsize(os.path.join(d, f)) > 100 for f in files_of(nm))

def push(nm):
    src, dst = os.path.join(SAVE_ROOT, nm), os.path.join(DEST_ROOT, nm)
    os.makedirs(dst, exist_ok=True)
    n_ok = n_bad = 0
    for f in sorted(os.listdir(src)):
        a, b = os.path.join(src, f), os.path.join(dst, f)
        try:
            if os.path.exists(b) and os.path.getsize(b) == os.path.getsize(a):
                n_ok += 1
                continue
            shutil.copy2(a, b)
            good = os.path.getsize(b) == os.path.getsize(a)
            n_ok += int(good); n_bad += int(not good)
        except Exception as e:
            n_bad += 1
            print('    gagal', f, e)
    print('  ke Drive: {} ok, {} gagal'.format(n_ok, n_bad))
    return n_bad == 0

@torch.no_grad()
def forward_all(model, ds):
    loader = tdata.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    out, ys, seen = [], [], 0
    t0 = time.time()
    for x, y in loader:
        out.append(model(x.cuda(non_blocking=True)).float().cpu())
        ys.append(y)
        seen += len(y)
        if seen % (BATCH_SIZE * 50) == 0:
            print('    {}/{} {:.0f}s'.format(seen, len(ds), time.time() - t0),
                  flush=True)
    return torch.cat(out).numpy(), torch.cat(ys).numpy().astype('int64')

for nm, expected in BACKBONES:
    print('\n' + '=' * 68)
    print(nm)
    print('=' * 68)
    d = os.path.join(SAVE_ROOT, nm)
    if in_drive(nm):
        META[nm] = json.load(open(os.path.join(DEST_ROOT, nm, 'meta.json')))
        print('  sudah lengkap di Drive | akurasi tercatat {:.4f}'.format(
            META[nm]['top1']))
        continue
    os.makedirs(d, exist_ok=True)
    tf, tf_desc = transform_for(nm)
    ds = torchvision.datasets.ImageFolder(VAL_DIR, tf)
    assert len(ds) == 50000
    model = load_model(nm, num_classes=1000, pretrained=True)
    hname, head = find_head(model)
    W = head.weight.detach().cpu().numpy().astype('float32')
    b = (head.bias.detach().cpu().numpy().astype('float32')
         if head.bias is not None else np.zeros(1000, 'float32'))
    print('  kepala: {} {} (bias: {})'.format(hname, tuple(W.shape),
                                             head.bias is not None))
    assert W.shape[0] == 1000, W.shape

    logits, y = forward_all(model, ds)
    top1 = float((logits.argmax(1) == y).mean())
    print('  top-1 {:.4f} | terbit {:.4f} | selisih {:+.4f}'.format(
        top1, expected, top1 - expected))
    assert abs(top1 - expected) < ACC_TOL, (
        'akurasi {:.4f} menyimpang > {} dari angka terbit {:.4f} -- transform atau '
        'bobotnya salah, dan tidak ada gejala lain yang akan muncul'.format(
            top1, ACC_TOL, expected))

    z = logits - logits.max(1, keepdims=True)
    p = np.exp(z); p /= p.sum(1, keepdims=True)
    np.save(d + '/logits.npy', logits.astype('float32'))
    np.save(d + '/scores.npy', p.astype('float32'))
    np.save(d + '/labels.npy', y)
    np.save(d + '/fc_weight.npy', W)
    np.save(d + '/fc_bias.npy', b)
    META[nm] = {'backbone': nm, 'top1': top1, 'top1_published': expected,
                'head_layer': hname, 'head_shape': list(W.shape),
                'has_bias': bool(head.bias is not None),
                'transform': tf_desc, 'n_rows': int(len(y)),
                'n_classes': int(logits.shape[1]),
                'classes_are_wnid_sorted': True,
                'scores_are': 'softmax(logits)', 'weights': 'IMAGENET1K_V1'}
    json.dump(META[nm], open(d + '/meta.json', 'w'), indent=1)
    push(nm)
    del model, logits, p, ds; gc.collect(); torch.cuda.empty_cache()

## 7. Verifikasi — dibaca kembali dari Drive

Bentuk, akurasi, dan **kecocokan label antar backbone**. Yang terakhir itu yang
penting: ketiga dump harus punya vektor label yang sama persis, kalau tidak
perbandingan antar backbone membandingkan urutan berkas, bukan model.

In [ ]:
ref_y = None
for nm, expected in BACKBONES:
    d = os.path.join(DEST_ROOT, nm)
    if not os.path.isdir(d):
        print('  {:14s} TIDAK ADA'.format(nm))
        continue
    S = np.load(d + '/scores.npy', mmap_mode='r')
    y = np.load(d + '/labels.npy')
    W = np.load(d + '/fc_weight.npy', mmap_mode='r')
    m = json.load(open(d + '/meta.json'))
    row = float(np.asarray(S[0]).sum())
    print('  {:14s} skor {} | W {} | top-1 {:.4f} | jumlah baris softmax {:.4f}'
          .format(nm, tuple(S.shape), tuple(W.shape), m['top1'], row))
    assert S.shape == (50000, 1000) and W.shape[0] == 1000
    assert abs(row - 1.0) < 1e-3, 'baris softmax tidak berjumlah 1'
    if ref_y is None:
        ref_y = y
    else:
        assert np.array_equal(ref_y, y), (
            nm + ': vektor label berbeda dari backbone pertama -- perbandingan '
            'antar backbone akan membandingkan urutan berkas, bukan model')

tot = sum(os.path.getsize(f) for f in glob.glob(DEST_ROOT + '/**/*', recursive=True)
          if os.path.isfile(f))
print('\nDrive: {:.2f} GB di {}'.format(tot / 1e9, DEST_ROOT))
print()
print('CAVEAT: dump ini 50.000 baris penuh; cache notebook 07 memakai protokol')
print('        25.000/25.000 milik paper UM-TTA. Jangan satukan keduanya di satu')
print('        tabel backbone -- itu membandingkan protokol, bukan backbone.')
print('CAVEAT: logits.npy ikut disimpan supaya suhu/kalibrasi bisa diubah nanti')
print('        tanpa GPU. scores.npy adalah softmax TANPA penskalaan suhu.')
print()
print('SELESAI. Notebook 12 membaca', DEST_ROOT, 'tanpa GPU.')